# 任务 1：数据加载与初步概览
* 导入所需库，读取数据集。
* 查看数据基本信息：df.info(), df.describe(), df.head()。
* 检查缺失值。

In [ ]:

# ==================== 任务1：数据加载与初步概览 ====================

def task1_load_and_overview():
    import pandas as pd
    import warnings
    warnings.filterwarnings("ignore")

    file_path = "winequality-red.csv"
    df = pd.read_csv(file_path, sep=";")

    print("=" * 50)
    print("任务1：数据加载与初步概览")
    print("=" * 50)

    print("\n>>> df.info()")
    df.info()

    print("\n>>> df.describe()")
    print(df.describe())

    print("\n>>> df.head()")
    print(df.head())

    print("\n>>> 缺失值统计")
    missing = df.isnull().sum()
    print(missing)
    print(f"总缺失值数量: {missing.sum()}")

    return df

df = task1_load_and_overview()


# 任务 2：可视化探索
* 绘制目标变量 quality 的直方图，观察评分分布。
* 绘制特征与目标的相关性热力图，初步筛选重要特征。
* 使用箱线图或散点图观察特征与质量评分的关系（例如酒精含量 vs 质量）。

In [ ]:

# ==================== 任务2：可视化探索 ====================

def task2_visual_exploration(df):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    plt.rcParams["font.sans-serif"] = ["SimHei"]
    plt.rcParams["axes.unicode_minus"] = False

    print("=" * 50)
    print("任务2：可视化探索")
    print("=" * 50)

    plt.figure(figsize=(8, 5))
    plt.hist(df["quality"], bins=10, edgecolor="black", alpha=0.75)
    plt.title("葡萄酒质量评分分布直方图")
    plt.xlabel("质量评分（quality）")
    plt.ylabel("样本数量")
    plt.tight_layout()
    plt.show()

    corr_matrix = df.corr(numeric_only=True)
    plt.figure(figsize=(11, 8))
    sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False)
    plt.title("特征与目标的相关性热力图")
    plt.tight_layout()
    plt.show()

    corr_to_quality = corr_matrix["quality"].sort_values(ascending=False)
    print("\n>>> 各特征与quality的相关系数（降序）：")
    print(corr_to_quality)

    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x="alcohol", y="quality", alpha=0.6)
    plt.title("alcohol 与 quality 的散点图")
    plt.xlabel("alcohol（酒精含量）")
    plt.ylabel("quality")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="quality", y="alcohol")
    plt.title("按 quality 分组的 alcohol 箱线图")
    plt.xlabel("quality")
    plt.ylabel("alcohol（酒精含量）")
    plt.tight_layout()
    plt.show()

    return corr_to_quality

corr_to_quality = task2_visual_exploration(df)


# 任务 3：特征工程
* 将 quality 转换为二分类标签 good（≥6 为 1，否则 0）。
* 划分特征矩阵 X 和两个目标向量 y_reg（原始评分）与 y_clf（二分类标签）。

In [ ]:

# ==================== 任务3：特征工程 ====================

def task3_feature_engineering(df):
    import pandas as pd

    print("=" * 50)
    print("任务3：特征工程")
    print("=" * 50)

    df_new = df.copy()
    df_new["good"] = (df_new["quality"] >= 6).astype(int)
    X = df_new.drop(columns=["quality", "good"])
    y_reg = df_new["quality"]
    y_clf = df_new["good"]

    print("\n>>> 二分类标签 good 分布：")
    print(y_clf.value_counts())
    print("\n>>> good 标签占比：")
    print(y_clf.value_counts(normalize=True).sort_index())
    print(f"\n特征矩阵X形状: {X.shape}")
    print(f"回归目标y_reg形状: {y_reg.shape}")
    print(f"分类目标y_clf形状: {y_clf.shape}")

    return df_new, X, y_reg, y_clf

df_new, X, y_reg, y_clf = task3_feature_engineering(df)


# 任务 4：数据划分与标准化
* 将数据划分为训练集（70%）和测试集（30%），设置随机种子保证可重复性。
* 对特征进行 Z-Score 标准化（StandardScaler），避免量纲影响。

In [ ]:

# ==================== 任务4：数据划分与标准化 ====================

def task4_split_and_scale(X, y_reg, y_clf):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    print("=" * 50)
    print("任务4：数据划分与标准化")
    print("=" * 50)

    X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
        X, y_reg, y_clf, test_size=0.3, random_state=42, stratify=y_clf
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"\n>>> 训练集样本数: {X_train.shape[0]}")
    print(f">>> 测试集样本数: {X_test.shape[0]}")
    print(f">>> 特征数量: {X_train.shape[1]}")
    print(f"\n>>> 标准化后训练集均值范围: [{X_train_scaled.mean(axis=0).min():.6f}, {X_train_scaled.mean(axis=0).max():.6f}]")
    print(f">>> 标准化后训练集标准差范围: [{X_train_scaled.std(axis=0).min():.6f}, {X_train_scaled.std(axis=0).max():.6f}]")

    split_dict = {
        "X_train": X_train,
        "X_test": X_test,
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "y_reg_train": y_reg_train,
        "y_reg_test": y_reg_test,
        "y_clf_train": y_clf_train,
        "y_clf_test": y_clf_test,
        "feature_names": X.columns.tolist(),
        "scaler": scaler,
    }

    return split_dict

split_dict = task4_split_and_scale(X, y_reg, y_clf)


# 任务 5：线性回归建模
* 使用 sklearn.linear_model.LinearRegression 训练模型。
* 在训练集上拟合，在测试集上预测。

In [ ]:

# ==================== 任务5：线性回归建模 ====================

def task5_linear_regression(split_dict):
    from sklearn.linear_model import LinearRegression

    print("=" * 50)
    print("任务5：线性回归建模")
    print("=" * 50)

    lr_model = LinearRegression()
    lr_model.fit(split_dict["X_train_scaled"], split_dict["y_reg_train"])

    print(f"\n>>> 模型截距（intercept）: {lr_model.intercept_:.4f}")
    print(f">>> 模型已训练完成，特征数量: {len(lr_model.coef_)}")

    return lr_model

lr_model = task5_linear_regression(split_dict)


# 任务 6：回归模型评估
* 计算评估指标：均方误差（MSE）、均方根误差（RMSE）、平均绝对误差（MAE）、决定系数（R²）。
* 绘制 残差图（预测值 vs 真实值），检查线性假设是否成立。
* 输出特征系数（coef_），分析哪些理化指标对酒质影响最大。

In [ ]:

# ==================== 任务6：回归模型评估 ====================

def task6_evaluate_regression(lr_model, split_dict):
    import numpy as np
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    import pandas as pd
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["SimHei"]
    plt.rcParams["axes.unicode_minus"] = False

    print("=" * 50)
    print("任务6：回归模型评估")
    print("=" * 50)

    y_pred = lr_model.predict(split_dict["X_test_scaled"])
    y_true = split_dict["y_reg_test"]

    mse_value = mean_squared_error(y_true, y_pred)
    rmse_value = np.sqrt(mse_value)
    mae_value = mean_absolute_error(y_true, y_pred)
    r2_value = r2_score(y_true, y_pred)

    print(f"\n>>> MSE  (均方误差)   : {mse_value:.4f}")
    print(f">>> RMSE (均方根误差)  : {rmse_value:.4f}")
    print(f">>> MAE  (平均绝对误差) : {mae_value:.4f}")
    print(f">>> R²   (决定系数)    : {r2_value:.4f}")

    residuals = y_true - y_pred
    plt.figure(figsize=(8, 5))
    plt.scatter(y_pred, residuals, alpha=0.6)
    plt.axhline(y=0, color="red", linestyle="--")
    plt.title("残差图（预测值 vs 残差）")
    plt.xlabel("预测值")
    plt.ylabel("残差（真实值 - 预测值）")
    plt.tight_layout()
    plt.show()

    coef_df = pd.DataFrame({
        "feature": split_dict["feature_names"],
        "coef": lr_model.coef_
    })
    coef_df["abs_coef"] = coef_df["coef"].abs()
    coef_df = coef_df.sort_values(by="abs_coef", ascending=False)

    print("\n>>> 特征系数（按绝对值降序，即对quality影响最大的特征）：")
    print(coef_df.to_string(index=False))

    return {
        "mse": mse_value,
        "rmse": rmse_value,
        "mae": mae_value,
        "r2": r2_value,
        "coef_df": coef_df,
        "y_pred": y_pred,
    }

reg_result = task6_evaluate_regression(lr_model, split_dict)


# 任务 7：分类数据准备
* 使用相同的标准化后的特征（X）和二分类标签 y_clf。注意：逻辑回归对特征尺度敏感，需标准化；决策树和 KNN 也建议标准化（KNN 尤其重要）。

In [ ]:

# ==================== 任务7：分类数据准备 ====================

def task7_classification_data_prep(split_dict):
    print("=" * 50)
    print("任务7：分类数据准备")
    print("=" * 50)

    print(f"\n>>> 训练集特征形状: {split_dict['X_train_scaled'].shape}")
    print(f">>> 测试集特征形状: {split_dict['X_test_scaled'].shape}")
    print(f">>> 训练集标签分布: {split_dict['y_clf_train'].value_counts().to_dict()}")
    print(f">>> 测试集标签分布: {split_dict['y_clf_test'].value_counts().to_dict()}")
    print("\n说明：分类任务使用与回归任务相同的标准化特征和划分方案，")
    print("      目标变量为二分类标签y_clf（good>=6为1，否则为0）。")

task7_classification_data_prep(split_dict)


# 任务 8：逻辑回归
* 使用 LogisticRegression 训练模型。
* 输出测试集上的准确率、精确率、召回率、F1 分数。
* 绘制 ROC 曲线，计算 AUC 值。

In [ ]:

# ==================== 任务8：逻辑回归 ====================

def task8_logistic_regression(split_dict):
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_curve, auc
    )
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["SimHei"]
    plt.rcParams["axes.unicode_minus"] = False

    print("=" * 50)
    print("任务8：逻辑回归")
    print("=" * 50)

    log_model = LogisticRegression(max_iter=300, random_state=42)
    log_model.fit(split_dict["X_train_scaled"], split_dict["y_clf_train"])
    y_pred = log_model.predict(split_dict["X_test_scaled"])
    y_prob = log_model.predict_proba(split_dict["X_test_scaled"])[:, 1]

    acc = accuracy_score(split_dict["y_clf_test"], y_pred)
    pre = precision_score(split_dict["y_clf_test"], y_pred, zero_division=0)
    rec = recall_score(split_dict["y_clf_test"], y_pred, zero_division=0)
    f1 = f1_score(split_dict["y_clf_test"], y_pred, zero_division=0)
    fpr, tpr, _ = roc_curve(split_dict["y_clf_test"], y_prob)
    auc_value = auc(fpr, tpr)

    print(f"\n>>> 准确率 (Accuracy) : {acc:.4f}")
    print(f">>> 精确率 (Precision) : {pre:.4f}")
    print(f">>> 召回率 (Recall)    : {rec:.4f}")
    print(f">>> F1分数 (F1-Score)  : {f1:.4f}")
    print(f">>> AUC值              : {auc_value:.4f}")

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"逻辑回归 AUC={auc_value:.4f}", linewidth=2)
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("假阳性率（FPR）")
    plt.ylabel("真阳性率（TPR）")
    plt.title("逻辑回归 ROC曲线")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    return log_model, {
        "model_name": "逻辑回归_默认",
        "accuracy": acc,
        "precision": pre,
        "recall": rec,
        "f1": f1,
        "auc": auc_value,
    }

log_model, logistic_metrics = task8_logistic_regression(split_dict)


# 任务 9：决策树
* 使用 DecisionTreeClassifier 训练，设置 max_depth 等参数防止过拟合。
* 输出分类评估指标。
*可视化决策树（可选）或输出特征重要性。

In [ ]:

# ==================== 任务9：决策树 ====================

def task9_decision_tree(split_dict):
    from sklearn.tree import DecisionTreeClassifier, plot_tree
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_curve, auc
    )
    import pandas as pd
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["SimHei"]
    plt.rcParams["axes.unicode_minus"] = False

    print("=" * 50)
    print("任务9：决策树")
    print("=" * 50)

    dt_model = DecisionTreeClassifier(
        max_depth=5, min_samples_split=10, random_state=42
    )
    dt_model.fit(split_dict["X_train_scaled"], split_dict["y_clf_train"])
    y_pred = dt_model.predict(split_dict["X_test_scaled"])
    y_prob = dt_model.predict_proba(split_dict["X_test_scaled"])[:, 1]

    acc = accuracy_score(split_dict["y_clf_test"], y_pred)
    pre = precision_score(split_dict["y_clf_test"], y_pred, zero_division=0)
    rec = recall_score(split_dict["y_clf_test"], y_pred, zero_division=0)
    f1 = f1_score(split_dict["y_clf_test"], y_pred, zero_division=0)
    fpr, tpr, _ = roc_curve(split_dict["y_clf_test"], y_prob)
    auc_value = auc(fpr, tpr)

    print(f"\n>>> 准确率 (Accuracy) : {acc:.4f}")
    print(f">>> 精确率 (Precision) : {pre:.4f}")
    print(f">>> 召回率 (Recall)    : {rec:.4f}")
    print(f">>> F1分数 (F1-Score)  : {f1:.4f}")
    print(f">>> AUC值              : {auc_value:.4f}")

    importance_df = pd.DataFrame({
        "feature": split_dict["feature_names"],
        "importance": dt_model.feature_importances_
    })
    importance_df = importance_df.sort_values(by="importance", ascending=False)
    print("\n>>> 决策树特征重要性（降序）：")
    print(importance_df.to_string(index=False))

    plt.figure(figsize=(20, 10))
    plot_tree(
        dt_model,
        feature_names=split_dict["feature_names"],
        class_names=["bad", "good"],
        filled=True,
        rounded=True,
        max_depth=3,
        fontsize=8,
    )
    plt.title("决策树可视化（仅展示前3层）")
    plt.tight_layout()
    plt.show()

    return dt_model, {
        "model_name": "决策树_默认",
        "accuracy": acc,
        "precision": pre,
        "recall": rec,
        "f1": f1,
        "auc": auc_value,
    }, importance_df

dt_model, tree_metrics, tree_importance_df = task9_decision_tree(split_dict)


# 任务 10：K 近邻（KNN）
* 使用 KNeighborsClassifier 训练，选择 K 值（如默认 5）。
* 评估分类性能。

In [ ]:

# ==================== 任务10：K近邻（KNN） ====================

def task10_knn(split_dict):
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_curve, auc
    )
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["SimHei"]
    plt.rcParams["axes.unicode_minus"] = False

    print("=" * 50)
    print("任务10：K近邻（KNN）")
    print("=" * 50)

    knn_model = KNeighborsClassifier(n_neighbors=5)
    knn_model.fit(split_dict["X_train_scaled"], split_dict["y_clf_train"])
    y_pred = knn_model.predict(split_dict["X_test_scaled"])
    y_prob = knn_model.predict_proba(split_dict["X_test_scaled"])[:, 1]

    acc = accuracy_score(split_dict["y_clf_test"], y_pred)
    pre = precision_score(split_dict["y_clf_test"], y_pred, zero_division=0)
    rec = recall_score(split_dict["y_clf_test"], y_pred, zero_division=0)
    f1 = f1_score(split_dict["y_clf_test"], y_pred, zero_division=0)
    fpr, tpr, _ = roc_curve(split_dict["y_clf_test"], y_prob)
    auc_value = auc(fpr, tpr)

    print(f"\n>>> 准确率 (Accuracy) : {acc:.4f}")
    print(f">>> 精确率 (Precision) : {pre:.4f}")
    print(f">>> 召回率 (Recall)    : {rec:.4f}")
    print(f">>> F1分数 (F1-Score)  : {f1:.4f}")
    print(f">>> AUC值              : {auc_value:.4f}")

    return knn_model, {
        "model_name": "KNN_默认",
        "accuracy": acc,
        "precision": pre,
        "recall": rec,
        "f1": f1,
        "auc": auc_value,
    }

knn_model, knn_metrics = task10_knn(split_dict)


# 任务 11：交叉验证与网格搜索
* 针对逻辑回归的 C 参数、决策树的 max_depth、KNN 的 n_neighbors，使用 GridSearchCV 进行 5 折交叉验证，寻找最优超参数。
* 对比调优前后模型在测试集上的性能变化。

In [ ]:

# ==================== 任务11：交叉验证与网格搜索 ====================

def task11_grid_search(split_dict, logistic_metrics, tree_metrics, knn_metrics):
    from sklearn.model_selection import GridSearchCV
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_curve, auc
    )
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["SimHei"]
    plt.rcParams["axes.unicode_minus"] = False

    print("=" * 50)
    print("任务11：交叉验证与网格搜索")
    print("=" * 50)

    X_train_scaled = split_dict["X_train_scaled"]
    y_train = split_dict["y_clf_train"]
    X_test_scaled = split_dict["X_test_scaled"]
    y_test = split_dict["y_clf_test"]

    logistic_grid = {
        "C": [0.01, 0.1, 1, 10, 100],
        "penalty": ["l2"],
        "solver": ["lbfgs"],
    }
    logistic_gs = GridSearchCV(
        estimator=LogisticRegression(max_iter=500, random_state=42),
        param_grid=logistic_grid,
        scoring="f1",
        cv=5,
        n_jobs=-1,
    )
    logistic_gs.fit(X_train_scaled, y_train)
    print(f"\n>>> 逻辑回归最优参数: {logistic_gs.best_params_}")
    print(f">>> 逻辑回归最优CV F1分数: {logistic_gs.best_score_:.4f}")

    dt_grid = {
        "max_depth": [3, 5, 7, None],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4, 8],
    }
    dt_gs = GridSearchCV(
        estimator=DecisionTreeClassifier(random_state=42),
        param_grid=dt_grid,
        scoring="f1",
        cv=5,
        n_jobs=-1,
    )
    dt_gs.fit(X_train_scaled, y_train)
    print(f"\n>>> 决策树最优参数: {dt_gs.best_params_}")
    print(f">>> 决策树最优CV F1分数: {dt_gs.best_score_:.4f}")

    knn_grid = {
        "n_neighbors": [3, 5, 7, 9, 11],
        "weights": ["uniform", "distance"],
    }
    knn_gs = GridSearchCV(
        estimator=KNeighborsClassifier(),
        param_grid=knn_grid,
        scoring="f1",
        cv=5,
        n_jobs=-1,
    )
    knn_gs.fit(X_train_scaled, y_train)
    print(f"\n>>> KNN最优参数: {knn_gs.best_params_}")
    print(f">>> KNN最优CV F1分数: {knn_gs.best_score_:.4f}")

    best_logistic = logistic_gs.best_estimator_
    best_tree = dt_gs.best_estimator_
    best_knn = knn_gs.best_estimator_

    y_pred_log = best_logistic.predict(X_test_scaled)
    y_prob_log = best_logistic.predict_proba(X_test_scaled)[:, 1]
    tuned_log_metrics = {
        "model_name": "逻辑回归_调优",
        "accuracy": accuracy_score(y_test, y_pred_log),
        "precision": precision_score(y_test, y_pred_log, zero_division=0),
        "recall": recall_score(y_test, y_pred_log, zero_division=0),
        "f1": f1_score(y_test, y_pred_log, zero_division=0),
        "auc": auc(*roc_curve(y_test, y_prob_log)[:2]),
    }

    y_pred_tree = best_tree.predict(X_test_scaled)
    y_prob_tree = best_tree.predict_proba(X_test_scaled)[:, 1]
    tuned_tree_metrics = {
        "model_name": "决策树_调优",
        "accuracy": accuracy_score(y_test, y_pred_tree),
        "precision": precision_score(y_test, y_pred_tree, zero_division=0),
        "recall": recall_score(y_test, y_pred_tree, zero_division=0),
        "f1": f1_score(y_test, y_pred_tree, zero_division=0),
        "auc": auc(*roc_curve(y_test, y_prob_tree)[:2]),
    }

    y_pred_knn = best_knn.predict(X_test_scaled)
    y_prob_knn = best_knn.predict_proba(X_test_scaled)[:, 1]
    tuned_knn_metrics = {
        "model_name": "KNN_调优",
        "accuracy": accuracy_score(y_test, y_pred_knn),
        "precision": precision_score(y_test, y_pred_knn, zero_division=0),
        "recall": recall_score(y_test, y_pred_knn, zero_division=0),
        "f1": f1_score(y_test, y_pred_knn, zero_division=0),
        "auc": auc(*roc_curve(y_test, y_prob_knn)[:2]),
    }

    metrics_names = ["Accuracy", "F1", "AUC"]
    models_names = ["逻辑回归", "决策树", "KNN"]
    default_vals = [
        [logistic_metrics["accuracy"], logistic_metrics["f1"], logistic_metrics["auc"]],
        [tree_metrics["accuracy"], tree_metrics["f1"], tree_metrics["auc"]],
        [knn_metrics["accuracy"], knn_metrics["f1"], knn_metrics["auc"]],
    ]
    tuned_vals = [
        [tuned_log_metrics["accuracy"], tuned_log_metrics["f1"], tuned_log_metrics["auc"]],
        [tuned_tree_metrics["accuracy"], tuned_tree_metrics["f1"], tuned_tree_metrics["auc"]],
        [tuned_knn_metrics["accuracy"], tuned_knn_metrics["f1"], tuned_knn_metrics["auc"]],
    ]
    x = np.arange(len(metrics_names))
    width = 0.12
    fig, ax = plt.subplots(figsize=(11, 6))
    colors_default = ["#4ECDC4", "#FFD93D", "#FF6B6B"]
    colors_tuned = ["#2E8B8B", "#C4A800", "#CC3333"]
    for i, model in enumerate(models_names):
        offset = (i - 1) * 2 * width
        ax.bar(x + offset, default_vals[i], width,
               label=f"{model} 默认", color=colors_default[i], edgecolor="black", linewidth=0.5)
        ax.bar(x + offset + width, tuned_vals[i], width,
               label=f"{model} 调优", color=colors_tuned[i], edgecolor="black", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_names)
    ax.set_ylabel("指标值")
    ax.set_title("调优前后性能对比")
    ax.legend(loc="lower right", fontsize=7)
    ax.set_ylim(0, 1.08)
    for patch in ax.patches:
        h = patch.get_height()
        if h > 0.01:
            ax.annotate(f"{h:.3f}", (patch.get_x() + patch.get_width() / 2, h),
                        ha="center", va="bottom", fontsize=7)
    plt.tight_layout()
    plt.show()

    best_models = {
        "logistic": best_logistic,
        "tree": best_tree,
        "knn": best_knn,
    }
    return best_models, tuned_log_metrics, tuned_tree_metrics, tuned_knn_metrics

best_models, tuned_log_metrics, tuned_tree_metrics, tuned_knn_metrics = task11_grid_search(
    split_dict, logistic_metrics, tree_metrics, knn_metrics
)


# 任务 12：模型综合对比
* 将三个分类模型（逻辑回归、决策树、KNN）的最优版本在同一测试集上比较：
+ 准确率、F1 分数、AUC 值。
+ 训练时间与预测时间。
* 绘制各模型的 混淆矩阵，直观展示分类错误类型。

In [ ]:

# ==================== 任务12：模型综合对比 ====================

def task12_model_comparison(best_models, split_dict):
    import time
    from sklearn.metrics import (
        accuracy_score, f1_score, roc_curve, auc,
        confusion_matrix, ConfusionMatrixDisplay
    )
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["SimHei"]
    plt.rcParams["axes.unicode_minus"] = False

    print("=" * 50)
    print("任务12：模型综合对比")
    print("=" * 50)

    X_train_scaled = split_dict["X_train_scaled"]
    X_test_scaled = split_dict["X_test_scaled"]
    y_train = split_dict["y_clf_train"]
    y_test = split_dict["y_clf_test"]

    model_map = {
        "逻辑回归_最优": best_models["logistic"],
        "决策树_最优": best_models["tree"],
        "KNN_最优": best_models["knn"],
    }

    records = []
    for model_name, model_obj in model_map.items():
        start_train = time.perf_counter()
        model_obj.fit(X_train_scaled, y_train)
        train_time = time.perf_counter() - start_train

        start_pred = time.perf_counter()
        y_pred = model_obj.predict(X_test_scaled)
        pred_time = time.perf_counter() - start_pred

        y_prob = model_obj.predict_proba(X_test_scaled)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_value = auc(fpr, tpr)

        records.append({
            "模型": model_name,
            "准确率": acc,
            "F1分数": f1,
            "AUC值": auc_value,
            "训练时间(秒)": train_time,
            "预测时间(秒)": pred_time,
        })

        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm, display_labels=["bad(0)", "good(1)"]
        )
        disp.plot(cmap="Blues")
        plt.title(f"{model_name} 混淆矩阵")
        plt.tight_layout()
        plt.show()

    benchmark_df = pd.DataFrame(records).sort_values(
        by=["F1分数", "AUC值"], ascending=False
    )

    metrics_cols = ["准确率", "F1分数", "AUC值"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(metrics_cols))
    width = 0.25
    colors = ["#4ECDC4", "#FFD93D", "#FF6B6B"]
    for i, (_, row) in enumerate(benchmark_df.iterrows()):
        vals = [row["准确率"], row["F1分数"], row["AUC值"]]
        bars = axes[0].bar(x + i * width, vals, width, label=row["模型"],
                           color=colors[i % 3], edgecolor="black", linewidth=0.5)
        for bar in bars:
            h = bar.get_height()
            if h > 0.01:
                axes[0].annotate(f"{h:.3f}", (bar.get_x() + bar.get_width() / 2, h),
                                 ha="center", va="bottom", fontsize=8)
    axes[0].set_xticks(x + width)
    axes[0].set_xticklabels(metrics_cols)
    axes[0].set_ylabel("指标值")
    axes[0].set_title("最优模型分类性能对比")
    axes[0].legend(loc="lower right", fontsize=9)
    axes[0].set_ylim(0, 1.08)

    time_cols = ["训练时间(秒)", "预测时间(秒)"]
    x2 = np.arange(len(time_cols))
    for i, (_, row) in enumerate(benchmark_df.iterrows()):
        vals2 = [row["训练时间(秒)"], row["预测时间(秒)"]]
        bars2 = axes[1].bar(x2 + i * width, vals2, width, label=row["模型"],
                            color=colors[i % 3], edgecolor="black", linewidth=0.5)
        for bar in bars2:
            h = bar.get_height()
            if h > 0.0001:
                axes[1].annotate(f"{h:.4f}", (bar.get_x() + bar.get_width() / 2, h),
                                 ha="center", va="bottom", fontsize=8)
    axes[1].set_xticks(x2 + width)
    axes[1].set_xticklabels(time_cols)
    axes[1].set_ylabel("时间（秒）")
    axes[1].set_title("最优模型运行时间对比")
    axes[1].legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.show()

    return benchmark_df

benchmark_df = task12_model_comparison(best_models, split_dict)


# 任务 13：结果分析
* 总结回归任务中影响葡萄酒质量的关键因素。
* 对比分类模型中哪种算法更适合本数据集，并给出理由。
* 讨论数据类别不平衡对模型的影响。

In [ ]:

# ==================== 任务13：结果分析 ====================

def task13_analysis(reg_result, benchmark_df, y_clf):
    import pandas as pd

    print("=" * 50)
    print("任务13：结果分析")
    print("=" * 50)

    coef_df = reg_result["coef_df"]
    print("\n>>> 回归任务中影响葡萄酒质量的关键因素（按系数绝对值降序TOP5）：")
    print(coef_df.head(5)[["feature", "coef"]].to_string(index=False))

    best_row = benchmark_df.iloc[0]
    print(f"\n>>> 分类算法推荐：{best_row['模型']}")
    print(f"    该模型在测试集上F1={best_row['F1分数']:.4f}, AUC={best_row['AUC值']:.4f}")
    class_ratio = y_clf.value_counts(normalize=True).sort_index()
    print(f"\n>>> 类别分布比例（0=bad, 1=good）：")
    print(class_ratio.to_dict())


task13_analysis(reg_result, benchmark_df, y_clf)


# 任务 14：扩展思考（可选，选做且正确加5分）
* 尝试使用其他回归模型（如岭回归、Lasso）并与线性回归对比。
* 尝试使用集成学习方法（随机森林、XGBoost）并观察性能提升。

In [ ]:

# ==================== 任务14：扩展思考 ====================

def task14_extensions(split_dict):
    from sklearn.linear_model import Ridge, Lasso
    from sklearn.metrics import r2_score, mean_squared_error
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (
        accuracy_score, f1_score, roc_curve, auc
    )
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["SimHei"]
    plt.rcParams["axes.unicode_minus"] = False

    print("=" * 50)
    print("任务14：扩展思考（可选）")
    print("=" * 50)

    ridge_model = Ridge(alpha=1.0, random_state=42)
    ridge_model.fit(split_dict["X_train_scaled"], split_dict["y_reg_train"])
    ridge_pred = ridge_model.predict(split_dict["X_test_scaled"])
    ridge_r2 = r2_score(split_dict["y_reg_test"], ridge_pred)
    ridge_rmse = np.sqrt(mean_squared_error(split_dict["y_reg_test"], ridge_pred))

    lasso_model = Lasso(alpha=0.001, random_state=42, max_iter=5000)
    lasso_model.fit(split_dict["X_train_scaled"], split_dict["y_reg_train"])
    lasso_pred = lasso_model.predict(split_dict["X_test_scaled"])
    lasso_r2 = r2_score(split_dict["y_reg_test"], lasso_pred)
    lasso_rmse = np.sqrt(mean_squared_error(split_dict["y_reg_test"], lasso_pred))

    reg_compare_df = pd.DataFrame([
        {"模型": "线性回归", "R²": reg_result["r2"], "RMSE": reg_result["rmse"]},
        {"模型": "岭回归", "R²": ridge_r2, "RMSE": ridge_rmse},
        {"模型": "Lasso回归", "R²": lasso_r2, "RMSE": lasso_rmse},
    ])

    reg_models = reg_compare_df["模型"].tolist()
    x = np.arange(len(reg_models))
    width = 0.35
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    bars_r2 = axes[0].bar(x, reg_compare_df["R²"], width,
                          color=["#4ECDC4", "#FFD93D", "#FF6B6B"], edgecolor="black", linewidth=0.5)
    for bar in bars_r2:
        h = bar.get_height()
        axes[0].annotate(f"{h:.4f}", (bar.get_x() + bar.get_width() / 2, h),
                         ha="center", va="bottom", fontsize=9)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(reg_models)
    axes[0].set_ylabel("R²")
    axes[0].set_title("回归模型 R^2 对比")
    bars_rmse = axes[1].bar(x, reg_compare_df["RMSE"], width,
                            color=["#4ECDC4", "#FFD93D", "#FF6B6B"], edgecolor="black", linewidth=0.5)
    for bar in bars_rmse:
        h = bar.get_height()
        axes[1].annotate(f"{h:.4f}", (bar.get_x() + bar.get_width() / 2, h),
                         ha="center", va="bottom", fontsize=9)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(reg_models)
    axes[1].set_ylabel("RMSE")
    axes[1].set_title("回归模型 RMSE 对比")
    plt.tight_layout()
    plt.show()

    rf_model = RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_split=5, random_state=42
    )
    rf_model.fit(split_dict["X_train_scaled"], split_dict["y_clf_train"])
    rf_pred = rf_model.predict(split_dict["X_test_scaled"])
    rf_acc = accuracy_score(split_dict["y_clf_test"], rf_pred)
    rf_f1 = f1_score(split_dict["y_clf_test"], rf_pred, zero_division=0)
    rf_prob = rf_model.predict_proba(split_dict["X_test_scaled"])[:, 1]
    rf_fpr, rf_tpr, _ = roc_curve(split_dict["y_clf_test"], rf_prob)
    rf_auc = auc(rf_fpr, rf_tpr)

    xgb_acc, xgb_f1, xgb_auc = None, None, None
    try:
        from xgboost import XGBClassifier
        xgb_model = XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            eval_metric="logloss", random_state=42,
        )
        xgb_model.fit(split_dict["X_train_scaled"], split_dict["y_clf_train"])
        xgb_pred = xgb_model.predict(split_dict["X_test_scaled"])
        xgb_acc = accuracy_score(split_dict["y_clf_test"], xgb_pred)
        xgb_f1 = f1_score(split_dict["y_clf_test"], xgb_pred, zero_division=0)
        xgb_prob = xgb_model.predict_proba(split_dict["X_test_scaled"])[:, 1]
        xgb_fpr, xgb_tpr, _ = roc_curve(split_dict["y_clf_test"], xgb_prob)
        xgb_auc = auc(xgb_fpr, xgb_tpr)
    except ImportError:
        print("\n注意：XGBoost未安装，跳过XGBoost评估。可使用 pip install xgboost 安装。")

    ensemble_compare_df = pd.DataFrame([
        {"模型": "随机森林", "准确率": rf_acc, "F1分数": rf_f1, "AUC值": rf_auc},
        {"模型": "XGBoost", "准确率": xgb_acc, "F1分数": xgb_f1, "AUC值": xgb_auc},
    ])

    ens_models = ensemble_compare_df["模型"].tolist()
    metrics_cols = ["准确率", "F1分数", "AUC值"]
    x = np.arange(len(metrics_cols))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["#4ECDC4", "#FFD93D"]
    for i, (_, row) in enumerate(ensemble_compare_df.iterrows()):
        vals = [row["准确率"], row["F1分数"], row["AUC值"]]
        bars = ax.bar(x + i * width, vals, width, label=row["模型"],
                      color=colors[i], edgecolor="black", linewidth=0.5)
        for bar in bars:
            h = bar.get_height()
            if h is not None and h > 0.01:
                ax.annotate(f"{h:.3f}", (bar.get_x() + bar.get_width() / 2, h),
                            ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(metrics_cols)
    ax.set_ylabel("指标值")
    ax.set_title("集成学习方法性能对比")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_ylim(0, 1.08)
    plt.tight_layout()
    plt.show()

    return {
        "ridge_r2": ridge_r2, "ridge_rmse": ridge_rmse,
        "lasso_r2": lasso_r2, "lasso_rmse": lasso_rmse,
        "rf_acc": rf_acc, "rf_f1": rf_f1, "rf_auc": rf_auc,
        "xgb_acc": xgb_acc, "xgb_f1": xgb_f1, "xgb_auc": xgb_auc,
    }

ext_result = task14_extensions(split_dict)
